In [82]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
import seaborn as sns
from sklearn.model_selection import KFold

from sklearn.inspection import permutation_importance
import time

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_squared_log_error
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import LabelEncoder, PolynomialFeatures, StandardScaler

In [51]:
train_df=pd.read_csv("./playground-series-s5e5/train.csv")
test_df=pd.read_csv("./playground-series-s5e5/test.csv")

In [53]:
test_ids=test_df['id']

In [55]:
numerical_features = ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp']

In [57]:
train=train_df.copy()
test=test_df.copy()

In [59]:
def add_feature_cross_terms(df, features):
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()]  
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            f1 = features[i]
            f2 = features[j]
            df[f"{f1}_x_{f2}"] = df[f1] * df[f2]
    return df

def add_interaction_features(df, features):
    df_new = df.copy()
    for f1, f2 in itertools.combinations(features, 2):
        df_new[f"{f1}_plus_{f2}"] = df_new[f1] + df_new[f2]
        df_new[f"{f1}_minus_{f2}"] = df_new[f1] - df_new[f2]
        df_new[f"{f2}_minus_{f1}"] = df_new[f2] - df_new[f1]
        df_new[f"{f1}_div_{f2}"] = df_new[f1] / (df_new[f2] + 1e-5)
        df_new[f"{f2}_div_{f1}"] = df_new[f2] / (df_new[f1] + 1e-5)
    return df_new

def add_statistical_features(df, features):
    df_new = df.copy()
    df_new["row_mean"] = df[features].mean(axis=1)
    df_new["row_std"] = df[features].std(axis=1)
    df_new["row_max"] = df[features].max(axis=1)
    df_new["row_min"] = df[features].min(axis=1)
    df_new["row_median"] = df[features].median(axis=1)
    return df_new

train = add_feature_cross_terms(train, numerical_features)
test = add_feature_cross_terms(test, numerical_features)

train = add_interaction_features(train, numerical_features)
test = add_interaction_features(test, numerical_features)

train = add_statistical_features(train, numerical_features)
test = add_statistical_features(test, numerical_features)

le = LabelEncoder()
train['Sex'] = le.fit_transform(train['Sex'])
test['Sex'] = le.transform(test['Sex'])

train['Sex'] = train['Sex'].astype('category')
test['Sex'] = test['Sex'].astype('category')

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_train = poly.fit_transform(train[numerical_features])
poly_test = poly.transform(test[numerical_features])
poly_feature_names = poly.get_feature_names_out(numerical_features)

poly_train_df = pd.DataFrame(poly_train, columns=poly_feature_names)
poly_test_df = pd.DataFrame(poly_test, columns=poly_feature_names)

train = pd.concat([train.reset_index(drop=True), poly_train_df], axis=1)
test = pd.concat([test.reset_index(drop=True), poly_test_df], axis=1)

In [61]:
X = train.drop(columns=['id', 'Calories'])
y = np.log1p(train['Calories'])
X_test = test.drop(columns=['id'])

FEATURES = X.columns.tolist()

In [63]:
print("X shape:", X.shape)
X.head()

X shape: (750000, 123)


,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Age_x_Height,Age_x_Weight,Age_x_Duration,...,Height Weight,Height Duration,Height Heart_Rate,Height Body_Temp,Weight Duration,Weight Heart_Rate,Weight Body_Temp,Duration Heart_Rate,Duration Body_Temp,Heart_Rate Body_Temp
0,1,36,189.0,82.0,26.0,101.0,41.0,6804.0,2952.0,936.0,...,15498.0,4914.0,19089.0,7749.0,2132.0,8282.0,3362.0,2626.0,1066.0,4141.0
1,0,64,163.0,60.0,8.0,85.0,39.7,10432.0,3840.0,512.0,...,9780.0,1304.0,13855.0,6471.1,480.0,5100.0,2382.0,680.0,317.6,3374.5
2,0,51,161.0,64.0,7.0,84.0,39.8,8211.0,3264.0,357.0,...,10304.0,1127.0,13524.0,6407.8,448.0,5376.0,2547.2,588.0,278.6,3343.2
3,1,20,192.0,90.0,25.0,105.0,40.7,3840.0,1800.0,500.0,...,17280.0,4800.0,20160.0,7814.4,2250.0,9450.0,3663.0,2625.0,1017.5,4273.5
4,0,38,166.0,61.0,25.0,102.0,40.6,6308.0,2318.0,950.0,...,10126.0,4150.0,16932.0,6739.6,1525.0,6222.0,2476.6,2550.0,1015.0,4141.2


### Removing duplicate features

In [68]:
def drop_duplicate_columns(df):
    hashes = df.apply(lambda col: pd.util.hash_pandas_object(col, index=False).sum())
    duplicated = hashes.duplicated(keep='first')
    return df.loc[:, ~duplicated]

In [72]:
X_cleaned=drop_duplicate_columns(X)

In [74]:
print("X_cleaned shape:", X_cleaned.shape)
X_cleaned.head()

X_cleaned shape: (750000, 102)


,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Age_x_Height,Age_x_Weight,Age_x_Duration,...,Heart_Rate_plus_Body_Temp,Heart_Rate_minus_Body_Temp,Body_Temp_minus_Heart_Rate,Heart_Rate_div_Body_Temp,Body_Temp_div_Heart_Rate,row_mean,row_std,row_min,row_median,Age
0,1,36,189.0,82.0,26.0,101.0,41.0,6804.0,2952.0,936.0,...,142.0,60.0,-60.0,2.463414,0.405941,79.166667,61.147090,26.0,61.50,36.0
1,0,64,163.0,60.0,8.0,85.0,39.7,10432.0,3840.0,512.0,...,124.7,45.3,-45.3,2.141057,0.467059,69.950000,52.482521,8.0,62.00,64.0
2,0,51,161.0,64.0,7.0,84.0,39.8,8211.0,3264.0,357.0,...,123.8,44.2,-44.2,2.110552,0.473809,67.800000,52.394656,7.0,57.50,51.0
3,1,20,192.0,90.0,25.0,105.0,40.7,3840.0,1800.0,500.0,...,145.7,64.3,-64.3,2.579852,0.387619,78.783333,65.466951,20.0,65.35,20.0
4,0,38,166.0,61.0,25.0,102.0,40.6,6308.0,2318.0,950.0,...,142.6,61.4,-61.4,2.512315,0.398039,72.100000,53.306472,25.0,50.80,38.0


In [76]:
X=X_cleaned

In [ ]:
"""import pandas as pd
import numpy as np

# Assume these are importance arrays of equal length
xgb_importance = xgb_model.feature_importances_
cat_importance = cat_model.get_feature_importance()
lgb_importance = lgb_model.feature_importance()

df = pd.DataFrame({
    'feature': X.columns,
    'xgb': xgb_importance,
    'cat': cat_importance,
    'lgb': lgb_importance
})

# Normalize importance (optional but helpful)
for col in ['xgb', 'cat', 'lgb']:
    df[col] = df[col] / df[col].max()

# Calculate average importance or rank
df['avg_importance'] = df[['xgb', 'cat', 'lgb']].mean(axis=1)
# Or use: df['avg_rank'] = df[['xgb', 'cat', 'lgb']].rank(ascending=False).mean(axis=1)

# Sort and get top features
top_features = df.sort_values(by='avg_importance', ascending=False)['feature'][:40].tolist()
print(top_features)
"""

In [84]:
FOLDS = 7
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)
models = {
    'CatBoost': CatBoostRegressor(verbose=100, random_seed=42, cat_features=['Sex'], early_stopping_rounds=100),
    'XGBoost': XGBRegressor(max_depth=10, colsample_bytree=0.7, subsample=0.9, n_estimators=2000, learning_rate=0.02,
                            gamma=0.01, max_delta_step=2, early_stopping_rounds=100, eval_metric='rmse',
                            enable_categorical=True, random_state=42),
    'LightGBM': LGBMRegressor(n_estimators=2000, learning_rate=0.02, max_depth=10, colsample_bytree=0.7,
                              subsample=0.9, random_state=42, verbose=-1)
}

results = {name: {'oof': np.zeros(len(train)), 'pred': np.zeros(len(test)), 'rmsle': []} for name in models}

for name, model in models.items():
    print(f"\n=== Training {name} ===")
    for i, (train_idx, valid_idx) in enumerate(kf.split(X, y)):
        print(f"\nFold {i+1}")
        x_train, y_train = X.iloc[train_idx], y[train_idx]
        x_valid, y_valid = X.iloc[valid_idx], y[valid_idx]
        
        x_train = x_train.loc[:, ~x_train.columns.duplicated()]
        x_valid = x_valid.loc[:, ~x_valid.columns.duplicated()]
        x_test = X_test.loc[:, ~X_test.columns.duplicated()].copy()

        start = time.time()
        
        if name == 'XGBoost':
            model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=100)
        elif name == 'CatBoost':
            model.fit(x_train, y_train, eval_set=(x_valid, y_valid))
        else:
            model.fit(x_train, y_train)

        oof_pred = model.predict(x_valid)
        test_pred = model.predict(x_test)
        
        results[name]['oof'][valid_idx] = oof_pred
        results[name]['pred'] += test_pred / FOLDS
        
        rmsle = np.sqrt(mean_squared_log_error(np.expm1(y_valid), np.expm1(oof_pred)))
        results[name]['rmsle'].append(rmsle)
        
        print(f"Fold {i+1} RMSLE: {rmsle:.4f}")
        print(f"Training time: {time.time() - start:.1f} sec")


print("\n=== Model Comparison ===")
for name in models:
    mean_rmsle = np.mean(results[name]['rmsle'])
    std_rmsle = np.std(results[name]['rmsle'])
    print(f"{name} - Mean RMSLE: {mean_rmsle:.4f} ± {std_rmsle:.4f}")


=== Training CatBoost ===

Fold 1
Learning rate set to 0.14053
0:	learn: 0.8366299	test: 0.8358985	best: 0.8358985 (0)	total: 41.3ms	remaining: 41.2s
100:	learn: 0.0632184	test: 0.0640517	best: 0.0640517 (100)	total: 3.45s	remaining: 30.7s
200:	learn: 0.0604332	test: 0.0620508	best: 0.0620508 (200)	total: 7.02s	remaining: 27.9s
300:	learn: 0.0590574	test: 0.0611536	best: 0.0611507 (298)	total: 10.2s	remaining: 23.7s
400:	learn: 0.0581642	test: 0.0606446	best: 0.0606446 (400)	total: 13.5s	remaining: 20.1s
500:	learn: 0.0575349	test: 0.0603967	best: 0.0603967 (500)	total: 16.7s	remaining: 16.6s
600:	learn: 0.0569786	test: 0.0602529	best: 0.0602499 (598)	total: 20s	remaining: 13.3s
700:	learn: 0.0565579	test: 0.0601519	best: 0.0601519 (700)	total: 23.4s	remaining: 9.97s
800:	learn: 0.0561505	test: 0.0600598	best: 0.0600597 (799)	total: 26.6s	remaining: 6.62s
900:	learn: 0.0558144	test: 0.0599909	best: 0.0599873 (892)	total: 29.8s	remaining: 3.28s
999:	learn: 0.0554836	test: 0.0599593	bes

ValueError: feature_names mismatch: ['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 'Age_x_Height', 'Age_x_Weight', 'Age_x_Duration', 'Age_x_Heart_Rate', 'Age_x_Body_Temp', 'Height_x_Weight', 'Height_x_Duration', 'Height_x_Heart_Rate', 'Height_x_Body_Temp', 'Weight_x_Duration', 'Weight_x_Heart_Rate', 'Weight_x_Body_Temp', 'Duration_x_Heart_Rate', 'Duration_x_Body_Temp', 'Heart_Rate_x_Body_Temp', 'Age_plus_Height', 'Age_minus_Height', 'Height_minus_Age', 'Age_div_Height', 'Height_div_Age', 'Age_plus_Weight', 'Age_minus_Weight', 'Weight_minus_Age', 'Age_div_Weight', 'Weight_div_Age', 'Age_plus_Duration', 'Age_minus_Duration', 'Duration_minus_Age', 'Age_div_Duration', 'Duration_div_Age', 'Age_plus_Heart_Rate', 'Age_minus_Heart_Rate', 'Heart_Rate_minus_Age', 'Age_div_Heart_Rate', 'Heart_Rate_div_Age', 'Age_plus_Body_Temp', 'Age_minus_Body_Temp', 'Body_Temp_minus_Age', 'Age_div_Body_Temp', 'Body_Temp_div_Age', 'Height_plus_Weight', 'Height_minus_Weight', 'Weight_mi